# POSEIDON 1D MEM | Increment 11
## In-Situ Stress Scenarios, Bounds and Depth Integration
**Mikael Elgo · Tier C · Uncalibrated educational screening**

This notebook produces compatible horizontal-stress experiments using inherited Sv/pressure scenarios. It verifies and extracts the cumulative release, runs the scientific workflow and all tests, presents QC, and exports results. No geographic SHmax direction or field-calibrated stress is supplied.

**Colab:** place `Poseidon_1D_MEM_Increment_11_v11.0.0.zip` in `MyDrive/Poseidon_1D_MEM/`, then **Run all**. Existing project folders remain intact. Typical local verification takes several minutes; Drive dependency installation may add time.


## 1. Select mode and locate the release
Review checks packaged results. Reproduce regenerates them from packaged derived inputs. Neither mode rereads private LAS files. Set PROJECT_ROOT only for an existing complete local extraction.


In [ ]:
from pathlib import Path
import os
import sys
import hashlib
import json
import subprocess

MODE = os.environ.get("P2MEM11_MODE", "review")
PROJECT_ROOT = os.environ.get("P2MEM11_PROJECT_ROOT", "")
ZIP_PATH = os.environ.get(
    "P2MEM11_ZIP_PATH",
    "/content/drive/MyDrive/Poseidon_1D_MEM/Poseidon_1D_MEM_Increment_11_v11.0.0.zip"
)


## 2. Safe extraction and checksum verification
The bootstrap is embedded so it can verify the project before importing any project modules. Missing ledgers, altered files, duplicate entries and unsafe paths are rejected.


In [ ]:
"""Standard-library-only safe extraction and file-ledger checks for Colab."""
import hashlib
from pathlib import Path, PurePosixPath
import shutil
import stat
import tempfile
from zipfile import ZipFile

LEDGER_NAME='INCREMENT_11_SHA256SUMS.txt'


def verify_tree(root):
    root=Path(root)
    if root.is_symlink() or not root.is_dir():raise ValueError('Project root must be a regular directory')
    seen=set();ledger=root/LEDGER_NAME
    if not ledger.is_file() or ledger.is_symlink():raise ValueError('Increment 11 ledger missing')
    for line in ledger.read_text(encoding='utf-8').splitlines():
        if not line or line.startswith('#'):continue
        try:digest,name=line.split('  ',1)
        except ValueError as exc:raise ValueError('Malformed ledger') from exc
        p=PurePosixPath(name)
        if (p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or p.as_posix()!=name
            or name in seen or not name or name==LEDGER_NAME):raise ValueError('Unsafe ledger path')
        if len(digest)!=64 or any(c not in '0123456789abcdef' for c in digest):raise ValueError('Malformed checksum')
        seen.add(name);target=root/name
        if any(root.joinpath(*p.parts[:i]).is_symlink() for i in range(1,len(p.parts)+1)):
            raise ValueError('Symlink in release path')
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest()!=digest:
            raise ValueError('Missing or changed release file: '+name)
    if not seen:raise ValueError('Empty ledger')
    return seen


def extract_release(archive,parent=None):
    archive=Path(archive)
    if not archive.is_file():raise FileNotFoundError('Set ZIP_PATH to the exact Increment 11 ZIP')
    parent=Path(parent) if parent is not None else Path(tempfile.gettempdir())
    parent.mkdir(parents=True,exist_ok=True);staging=None
    try:
        with ZipFile(archive) as z:
            names=set();total=0
            for info in z.infolist():
                name=info.filename;p=PurePosixPath(name);total+=info.file_size
                if (p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or p.as_posix()!=name
                    or name in names or not name or info.is_dir() or stat.S_ISLNK(info.external_attr>>16)):
                    raise ValueError('Unsafe or duplicate ZIP entry: '+name)
                names.add(name)
            if LEDGER_NAME not in names:raise ValueError('Incomplete ZIP: release ledger missing; download the complete Increment 11 release')
            if total>300_000_000 or len(names)>10000:raise ValueError('Unexpected package size')
            staging=Path(tempfile.mkdtemp(prefix='poseidon_inc11_',dir=parent))
            z.extractall(staging)
        listed=verify_tree(staging)
        if names!=listed|{LEDGER_NAME}:raise ValueError('ZIP and ledger inventories differ')
        return staging
    except BaseException:
        if staging is not None:shutil.rmtree(staging,ignore_errors=True)
        raise


In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
if MODE not in ("review", "reproduce"):
    raise ValueError("MODE must be review or reproduce")
if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT).expanduser().resolve()
    verify_tree(project_root)
else:
    archive = Path(ZIP_PATH).expanduser().resolve()
    if not archive.is_file():
        raise FileNotFoundError(f"Upload the Increment 11 ZIP to this path or set ZIP_PATH:\n{archive}")
    print("Archive SHA-256:", hashlib.sha256(archive.read_bytes()).hexdigest())
    project_root = extract_release(archive, "/content" if IN_COLAB else None)
print("Verified working folder:", project_root)
print("Verified ledger entries:", len(verify_tree(project_root)))


## 3. Dependencies and cumulative verification
Tests run in an isolated copy. The independent checker solves the compliance system and uses Mohr-circle geometry. A test failure stops the notebook.


In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(project_root) + "[dev]"], check=True)
command = [sys.executable, str(project_root / "scripts/run_increment_11.py"), "--mode", MODE]
result = subprocess.run(command, cwd=project_root, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode:
    raise RuntimeError("Increment 11 verification failed. Read the error above before proceeding.")
verify_tree(project_root)
report = json.loads((project_root / "run_records/increment_11_run.json").read_text())
assert report["independent"]["passed"]
print("PASS:", report["tests_passed"], "tests;", report["independent"]["checked_numeric_rows"], "independent numeric rows")


## 4. Scientific model
Compression is positive; stresses use MPa. At zero horizontal strain:

$$S_{h0}=\frac{\nu}{1-\nu}(S_v-\alpha P_p)+\alpha P_p.$$

Imposed strain adds $E(\epsilon_x+\nu\epsilon_y)/(1-\nu^2)$ and its counterpart. E is required only for nonzero-strain cases. Here static nu is assumed and E is an unverified analogue additionally treated as drained equivalent.

Fault friction uses **S-Pp**, independently of intact-rock Biot alpha. The conditional principal-stress ratio limit is $(\sqrt{1+\mu^2}+\mu)^2$. Negative/nonpositive effective states and violated friction bounds are retained with rejection flags. No stress is clipped to a bound. See the scientific report for sources, parameter assumptions and all stopping conditions.


In [ ]:
output_dir = project_root / "outputs/11_horizontal_stress"
manifest = json.loads((output_dir / "stress_manifest.json").read_text())
print(json.dumps(manifest["coverage"], indent=2))
print("Scenario configuration:")
print((project_root / "config/horizontal_stress_scenarios.json").read_text())


## 5. Read the outputs
The join uses 202 exact source reporting nodes, preserving original LAS indices and the inherited depth basis. It does not create a continuous full-depth stress profile. Poseidon North 1 and Proteus remain blocked by absolute-Sv scenario availability.

The diagrams show selected nodes and terminal-node polygons. Polygon coordinates and depths are in `stress_polygon_vertices.csv`. Polygon interiors depend on assumed fault friction and reference pressure; they do not identify a unique SHmax. Equal horizontal stresses leave their azimuth undetermined.


In [ ]:
from IPython.display import display, Image
for well in ("Poseidon_2", "Boreas_1"):
    display(Image(filename=str(output_dir / (well + "_stress_qc.png"))))


## 6. Handoff and export
Pass whole scenario tuples to subsequent work. Field eligibility remains false even for cases passing the selected friction test. Increment 12 must add independent strength, trajectory and relative-orientation checks before any conditional wellbore-wall calculations.

The exported results ZIP includes tables, figures, methods, documentation and this run's verification record. It is a results bundle, not the cumulative software release.


In [ ]:
import zipfile
results_zip = project_root.parent / "Poseidon_Increment_11_Results.zip"
with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(output_dir.iterdir()):
        z.write(p, "results/" + p.name)
    for name in ("INCREMENT_11_SCIENTIFIC_REPORT.md", "INCREMENT_11_QUICKSTART.md",
                 "config/horizontal_stress_scenarios.json", "config/horizontal_stress_methods.json",
                 "run_records/increment_11_run.json"):
        z.write(project_root / name, name)
with zipfile.ZipFile(results_zip) as z:
    assert z.testzip() is None
print("Results:", results_zip)
print("SHA-256:", hashlib.sha256(results_zip.read_bytes()).hexdigest())
if IN_COLAB:
    from google.colab import files
    files.download(str(results_zip))
